# LeetCode 869 — Reordered Power of 2 bằng Python

Link đề: https://leetcode.com/problems/reordered-power-of-2/ · Độ khó: **Medium** · Chủ đề: Hash Table (bảng băm), Math (toán), Counting (đếm), Enumeration (liệt kê), Sorting

Skeleton LeetCode:

```python
class Solution:
    def reorderedPowerOf2(self, n: int) -> bool:
        ...
```

Ghi chú: hoán vị chữ số của `n` (không cho chữ số 0 đứng đầu) để được một **lũy thừa của 2**. Với `n <= 10^9` có tới `10!` hoán vị, nhưng chỉ có **~30 lũy thừa của 2** trong phạm vi. Đảo ngược cách nhìn: thay vì thử mọi hoán vị của `n`, ta so sánh **"dấu vân tay" chữ số** của `n` với của từng lũy thừa của 2. Hai số là hoán vị của nhau khi và chỉ khi chúng có **cùng số lần xuất hiện của mỗi chữ số**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: được xếp lại các chữ số của `n` theo thứ tự bất kỳ (chữ số đầu khác 0) và hỏi có thể được lũy thừa của 2 không.
2. Nắm các công cụ Python cần thiết: `itertools.permutations` (và `set` để khử trùng), đếm chữ số bằng `divmod` hoặc `collections.Counter`, chuỗi chữ số đã sắp xếp `"".join(sorted(str(n)))`, kiểm tra lũy thừa của 2 bằng `x & (x - 1) == 0`, và dịch bit `1 << i`.
3. Hiểu nguyên lý: **hai số là hoán vị của nhau ⇔ cùng đa tập chữ số** (cùng chữ số sắp xếp, cùng bảng đếm) và biến mọi so sánh hoán vị thành so sánh một "dấu vân tay".
4. Đi qua 5 cách: thử mọi hoán vị, quay lui theo bảng đếm có khử trùng, so bảng đếm với từng lũy thừa, tập các chuỗi chữ số đã sắp xếp, dấu vân tay bằng **tích các số nguyên tố**; đối chiếu bằng oracle sinh mọi hoán vị của các lũy thừa.
5. Tránh các lỗi kinh điển: cho phép chữ số 0 đứng đầu, chỉ xét lũy thừa `<= n`, dùng tập (set) thay vì đa tập chữ số, dùng dấu vân tay quá yếu (tổng chữ số), và quên các lũy thừa dài tới 10 chữ số.

## 2. Đề bài & ví dụ

Cho số nguyên `n`. Ta xếp lại các chữ số của `n` theo **thứ tự bất kỳ** (kể cả thứ tự ban đầu) sao cho **chữ số đầu tiên khác 0**.

Trả về `true` nếu và chỉ nếu có thể làm như vậy để số thu được là **lũy thừa của 2**.

**Ví dụ 1**

- Input: `n = 1`
- Output: `true` (`1 = 2^0`)

**Ví dụ 2**

- Input: `n = 10`
- Output: `false`
- Giải thích: hoán vị `10` và `01`; `01` có chữ số 0 đứng đầu nên không hợp lệ, còn `10` không là lũy thừa của 2.

**Ràng buộc:**

- `1 <= n <= 10^9`

Ví dụ thêm: `n = 46` -> `true` (hoán vị `64 = 2^6`); `n = 4201` -> `true` (hoán vị `1024 = 2^10`).

In [1]:
from typing import List
from itertools import permutations
from collections import Counter
import random, time

examples = [
    (1, True),
    (10, False),
]
extra = [(46, True), (4201, True), (3, False)]
for n, expected in examples + extra:
    print(f"n={n} kỳ vọng={expected}")

n=1 kỳ vọng=True
n=10 kỳ vọng=False
n=46 kỳ vọng=True
n=4201 kỳ vọng=True
n=3 kỳ vọng=False


## 3. Kiến thức nền

### 3.1. Hoán vị và hoán vị trùng lặp

`itertools.permutations(s)` sinh mọi hoán vị **theo vị trí**: chuỗi `"112"` cho `3! = 6` kết quả nhưng chỉ có `3` chuỗi **khác nhau** (`112, 121, 211`). Dùng `set` để khử trùng.

In [2]:
perms = list(permutations("112"))
print(len(perms), "hoán vị theo vị trí;", len(set("".join(p) for p in perms)), "chuỗi khác nhau:", sorted(set("".join(p) for p in perms)))

6 hoán vị theo vị trí; 3 chuỗi khác nhau: ['112', '121', '211']


### 3.2. Kiểm tra lũy thừa của 2

`x` là lũy thừa của 2 khi `x > 0` và `x & (x - 1) == 0`: lũy thừa của 2 chỉ có **một bit 1** trong biểu diễn nhị phân; trừ 1 làm bit đó thành 0 và mọi bit thấp thành 1, nên phép `&` bằng 0. Cũng có thể liệt kê `1 << i`.

In [3]:
def is_pow2(x: int) -> bool:
    return x > 0 and x & (x - 1) == 0

print([x for x in range(1, 70) if is_pow2(x)])
print([1 << i for i in range(11)])
print("các lũy thừa của 2 từ 2^0 tới 2^33 có số chữ số:", [len(str(1 << i)) for i in range(34)])

[1, 2, 4, 8, 16, 32, 64]
[1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
các lũy thừa của 2 từ 2^0 tới 2^33 có số chữ số: [1, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 7, 8, 8, 8, 9, 9, 9, 10, 10, 10, 10]


### 3.3. Đếm chữ số và "dấu vân tay"

Hai số là **hoán vị chữ số của nhau** khi và chỉ khi mọi chữ số `0..9` xuất hiện **cùng số lần** trong hai số. Có ba cách biểu diễn "dấu vân tay" tương đương:

- danh sách đếm 10 phần tử `cnt[d]`;
- chuỗi chữ số đã sắp xếp `"".join(sorted(str(x)))`;
- một số duy nhất, ví dụ **tích các số nguyên tố**: ánh xạ chữ số `d` thành số nguyên tố thứ `d` (`2, 3, 5, 7, 11, 13, 17, 19, 23, 29`) rồi nhân; theo **định lý cơ bản của số học** (phân tích ra thừa số nguyên tố duy nhất) hai đa tập chữ số khác nhau cho tích khác nhau.

In [4]:
def digit_count(x: int) -> List[int]:
    cnt = [0] * 10
    while x:
        x, d = divmod(x, 10)
        cnt[d] += 1
    return cnt

PRIMES = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
def prime_product(x: int) -> int:
    prod = 1
    while x:
        x, d = divmod(x, 10)
        prod *= PRIMES[d]
    return prod

for x in (1024, 4201, 2014):
    print(f"{x}: đếm {digit_count(x)} | sắp xếp {''.join(sorted(str(x)))} | tích nguyên tố {prime_product(x)}")
print("1024 và 4201 là hoán vị của nhau:", digit_count(1024) == digit_count(4201) == digit_count(2014))

1024: đếm [1, 1, 1, 0, 1, 0, 0, 0, 0, 0] | sắp xếp 0124 | tích nguyên tố 330
4201: đếm [1, 1, 1, 0, 1, 0, 0, 0, 0, 0] | sắp xếp 0124 | tích nguyên tố 330
2014: đếm [1, 1, 1, 0, 1, 0, 0, 0, 0, 0] | sắp xếp 0124 | tích nguyên tố 330
1024 và 4201 là hoán vị của nhau: True


### 3.4. Chữ số 0 đứng đầu

Hoán vị `"01"` của `10` biểu diễn `1` nhưng **chữ số đầu là 0 nên không được phép**. Khi liệt kê hoán vị phải bỏ các hoán vị bắt đầu bằng `'0'`. Ngược lại khi so sánh bảng đếm với một lũy thừa của 2 thì **không cần lo**: các lũy thừa của 2 không có chữ số 0 đứng đầu (biểu diễn thập phân chuẩn), và khớp bảng đếm với lũy thừa luôn cho một hoán vị có chữ số đầu khác 0 (vì ta chọn đúng hoán vị là chính lũy thừa đó).

In [5]:
n = 10
all_perms = {"".join(p) for p in permutations(str(n))}
print("hoán vị:", sorted(all_perms), "| hợp lệ (không bắt đầu bằng 0):", sorted(p for p in all_perms if p[0] != "0"))
print("hoán vị bắt đầu bằng 0 có thể là lũy thừa của 2:", [p for p in all_perms if is_pow2(int(p))])

hoán vị: ['01', '10'] | hợp lệ (không bắt đầu bằng 0): ['10']
hoán vị bắt đầu bằng 0 có thể là lũy thừa của 2: ['01']


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Đáp án là `true` khi tồn tại lũy thừa của 2 nào đó **cùng đa tập chữ số** với `n`.

- `n = 46`: chữ số `{4, 6}`; `64 = 2^6` có chữ số `{6, 4}` ⇒ `true`.
- `n = 4201`: chữ số `{4, 2, 0, 1}`; `1024 = 2^10` có chữ số `{1, 0, 2, 4}` ⇒ `true`.
- `n = 10`: chữ số `{1, 0}`; các lũy thừa 2 chữ số là `16, 32, 64`; không có `{1, 0}` ⇒ `false`. (Hoán vị `01 = 1` là lũy thừa nhưng chữ số 0 đứng đầu, và `1` có độ dài 1.)

Chỉ cần so với các lũy thừa của 2 **cùng độ dài** (cùng số chữ số) với `n`; có tối đa 4 lũy thừa cho mỗi độ dài.

In [6]:
powers_by_len = {}
for i in range(34):
    p = 1 << i
    powers_by_len.setdefault(len(str(p)), []).append(p)
for L in range(1, 11):
    print(f"{L:2} chữ số: {powers_by_len[L]}")

 1 chữ số: [1, 2, 4, 8]
 2 chữ số: [16, 32, 64]
 3 chữ số: [128, 256, 512]
 4 chữ số: [1024, 2048, 4096, 8192]
 5 chữ số: [16384, 32768, 65536]
 6 chữ số: [131072, 262144, 524288]
 7 chữ số: [1048576, 2097152, 4194304, 8388608]
 8 chữ số: [16777216, 33554432, 67108864]
 9 chữ số: [134217728, 268435456, 536870912]
10 chữ số: [1073741824, 2147483648, 4294967296, 8589934592]


## 5. Cách 1 — Brute Force: thử mọi hoán vị

**Ý tưởng.** Sinh mọi hoán vị của các chữ số của `n` (khử trùng bằng `set`), bỏ hoán vị bắt đầu bằng `'0'`, và kiểm tra có là lũy thừa của 2 không. Đây là định nghĩa trực tiếp nên cũng là **oracle**.

In [7]:
def reordered_brute(n: int) -> bool:
    digits = str(n)
    for perm in set(permutations(digits)):
        if perm[0] == "0":
            continue
        if is_pow2(int("".join(perm))):
            return True
    return False

for n, expected in examples + extra:
    got = reordered_brute(n)
    print(n, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

1 -> True OK
10 -> False OK
46 -> True OK
4201 -> True OK
3 -> False OK


**Độ phức tạp và nút thắt.** `O(L!)` hoán vị cho `L` chữ số (tới `10! = 3.628.800` khi `n` có 10 chữ số khác nhau), mỗi hoán vị `O(L)`: với `n = 1234567890` là hàng chục triệu thao tác, và còn tạo set rất lớn. Hãy xem thời gian:

In [8]:
for n in (123, 1234, 12345, 123456, 1234567, 12345678, 123456789):
    t0 = time.perf_counter()
    reordered_brute(n)
    print(f"n={n:>10} ({len(str(n))} chữ số khác nhau): {1000 * (time.perf_counter() - t0):9.1f} ms")

n=       123 (3 chữ số khác nhau):       0.0 ms
n=      1234 (4 chữ số khác nhau):       0.0 ms
n=     12345 (5 chữ số khác nhau):       0.0 ms
n=    123456 (6 chữ số khác nhau):       0.7 ms
n=   1234567 (7 chữ số khác nhau):       3.0 ms
n=  12345678 (8 chữ số khác nhau):      33.4 ms


n= 123456789 (9 chữ số khác nhau):     549.3 ms


Thời gian tăng gần **gấp `L` lần mỗi chữ số thêm**. Với 10 chữ số sẽ mất hàng chục giây. Nút thắt: ta **thử mọi hoán vị** trong khi chỉ có **~30 lũy thừa của 2** để khớp.

## 6. Ý tưởng thuật toán / bước đột phá

**Đảo ngược cách nhìn.** Thay vì sinh hoán vị của `n` rồi kiểm tra "có là lũy thừa 2", ta duyệt các **ứng viên đích** (các lũy thừa `2^0 .. 2^33`, chỉ khoảng 34 số, mỗi độ dài có `<= 4`) và kiểm tra "`n` có là hoán vị của lũy thừa đó không", tức so sánh **đa tập chữ số**. Phép so sánh này `O(L)` thay vì `O(L!)`.

Các cách biểu diễn dấu vân tay khác nhau cho ba cách giải:

| Cách | Ý tưởng |
|---|---|
| 2 | Quay lui theo **bảng đếm** chữ số (tránh hoán vị trùng); vẫn là vét cạn nhưng nhanh hơn Cách 1 |
| 3 | So **bảng đếm** của `n` với từng lũy thừa của 2 (lời giải chuẩn) |
| 4 | Tập các **chuỗi chữ số đã sắp xếp** của mọi lũy thừa; tra `n` bằng `in` |
| 5 | Dấu vân tay là **tích số nguyên tố**; tập các tích của mọi lũy thừa |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Quay lui theo bảng đếm chữ số

**Ý tưởng khác.** Vẫn là vét cạn nhưng không sinh hoán vị trùng: giữ `cnt[d]` là số chữ số `d` còn lại; dựng số từ trái sang phải, mỗi bước chọn một chữ số `d` còn dư (`cnt[d] > 0`); chữ số **đầu** phải khác 0. Khi đủ `L` chữ số thì kiểm tra lũy thừa của 2.

**Giải thích từng đoạn.**

- `dfs(pos, value)`: `pos` là số chữ số đã dựng, `value` là giá trị hiện tại.
- Với mỗi `d` từ `0..9` có `cnt[d] > 0` (và `d != 0` nếu `pos == 0`): trừ `cnt[d]`, đệ quy với `value * 10 + d`, rồi khôi phục.
- Mỗi số **khác nhau** được sinh đúng một lần (không có trùng như `permutations`).

In [9]:
def reordered_dfs(n: int) -> bool:
    cnt = digit_count(n)
    L = len(str(n))

    def dfs(pos: int, value: int) -> bool:
        if pos == L:
            return is_pow2(value)
        for d in range(10):
            if cnt[d] == 0 or (pos == 0 and d == 0):
                continue
            cnt[d] -= 1
            if dfs(pos + 1, value * 10 + d):
                return True
            cnt[d] += 1
        return False

    return dfs(0, 0)

for n, expected in examples + extra:
    assert reordered_dfs(n) == expected
print("quay lui theo bảng đếm OK")

quay lui theo bảng đếm OK


**Code nộp LeetCode (Cách 2):** đúng nhưng vẫn có thể chậm với 10 chữ số khác nhau (`~3,6 triệu` số); chậm nhất là vài giây ở trường hợp xấu.

In [10]:
class Solution:
    def reorderedPowerOf2(self, n: int) -> bool:
        cnt = [0] * 10
        m, L = n, 0
        while m:
            m, d = divmod(m, 10)
            cnt[d] += 1
            L += 1

        def dfs(pos: int, value: int) -> bool:
            if pos == L:
                return value & (value - 1) == 0
            for d in range(10):
                if cnt[d] == 0 or (pos == 0 and d == 0):
                    continue
                cnt[d] -= 1
                if dfs(pos + 1, value * 10 + d):
                    return True
                cnt[d] += 1
            return False

        return dfs(0, 0)

sol_dfs = Solution().reorderedPowerOf2
print([sol_dfs(n) for n, _ in examples + extra])

[True, False, True, True, False]


**Độ phức tạp (Cách 2):** `O(số số khác nhau * L)`, tệ nhất `~10! * 10` thao tác; bộ nhớ `O(L)` (ngăn xếp đệ quy).

---

### 7.2. Cách 3 — So bảng đếm với từng lũy thừa của 2 (lời giải chuẩn)

**Giải thích từng đoạn.**

- `f(x)` trả về bảng đếm 10 phần tử của chữ số `x`.
- `target = f(n)`.
- `i = 1`, lặp `i <= 10^9` (lũy thừa của 2 tới `2^29 = 536870912`); nếu `f(i) == target` thì `True`; `i <<= 1`.
- Hết vòng lặp: `False`.

Lưu ý: `n <= 10^9` có tối đa 10 chữ số (chỉ `n = 10^9` có 10 chữ số, gồm `1` và chín số `0`); bảng đếm của nó không khớp lũy thừa nào, nên chặn `i <= 10^9` đủ. Để an toàn với mọi độ dài, ta cũng kiểm tra bản mở rộng `i < 10^10` ở phần tính đúng.

In [11]:
def reordered_count(n: int) -> bool:
    target = digit_count(n)
    i = 1
    while i <= 10 ** 9:
        if digit_count(i) == target:
            return True
        i <<= 1
    return False

for n, expected in examples + extra:
    assert reordered_count(n) == expected
print("so bảng đếm OK")

so bảng đếm OK


**Code nộp LeetCode (Cách 3):**

In [12]:
class Solution:
    def reorderedPowerOf2(self, n: int) -> bool:
        def f(x: int) -> list:
            cnt = [0] * 10
            while x:
                x, v = divmod(x, 10)
                cnt[v] += 1
            return cnt

        target = f(n)
        i = 1
        while i <= 10**9:
            if f(i) == target:
                return True
            i <<= 1
        return False

sol_count = Solution().reorderedPowerOf2
print([sol_count(n) for n, _ in examples + extra])

[True, False, True, True, False]


**Độ phức tạp (Cách 3):** `O(log^2 M)` với `M = 10^9`: khoảng 30 lũy thừa, mỗi lần tính bảng đếm `O(10)` chữ số; bộ nhớ `O(1)`.

---

### 7.3. Cách 4 — Tập các chuỗi chữ số đã sắp xếp

**Ý tưởng khác.** Dấu vân tay là **chuỗi chữ số đã sắp xếp**: `"".join(sorted(str(x)))`. Tính trước tập `S` của dấu vân tay của mọi lũy thừa của 2 `<= 10^10`; với `n` chỉ cần `"".join(sorted(str(n))) in S` (một lần băm).

**Giải thích từng đoạn.** Tập `S` chỉ có khoảng 34 phần tử (một cho mỗi lũy thừa; một số lũy thừa có thể trùng dấu vân tay, khi đó tập nhỏ hơn).

In [13]:
POW_SIGS = {"".join(sorted(str(1 << i))) for i in range(34)}
print("số dấu vân tay khác nhau:", len(POW_SIGS), "| ví dụ:", sorted(POW_SIGS)[:6])

def reordered_sorted_set(n: int) -> bool:
    return "".join(sorted(str(n))) in POW_SIGS

for n, expected in examples + extra:
    assert reordered_sorted_set(n) == expected
print("tập chuỗi sắp xếp OK")

số dấu vân tay khác nhau: 34 | ví dụ: ['0112344778', '011237', '0122579', '012356789', '0124', '0134449']
tập chuỗi sắp xếp OK


**Code nộp LeetCode (Cách 4):**

In [14]:
class Solution:
    def reorderedPowerOf2(self, n: int) -> bool:
        sigs = {"".join(sorted(str(1 << i))) for i in range(34)}
        return "".join(sorted(str(n))) in sigs

sol_sorted = Solution().reorderedPowerOf2
print([sol_sorted(n) for n, _ in examples + extra])

[True, False, True, True, False]


**Độ phức tạp (Cách 4):** tiền xử lý `O(34 * 10 log 10)`; mỗi truy vấn `O(L log L)` sắp xếp chữ số; bộ nhớ `O(34 * 10)`.

---

### 7.4. Cách 5 — Dấu vân tay bằng tích các số nguyên tố

**Ý tưởng khác.** Gán chữ số `d` cho số nguyên tố thứ `d` (`0 -> 2, 1 -> 3, 2 -> 5, ..., 9 -> 29`) và lấy **tích** các số nguyên tố của mọi chữ số. Theo định lý phân tích ra thừa số nguyên tố duy nhất, hai số có cùng tích khi và chỉ khi có **cùng đa tập chữ số**. Tích là **một số nguyên** (không cần cấu trúc dữ liệu), cho phép so sánh và băm cực nhanh.

**Giải thích từng đoạn.** Tích tối đa `29^10 ≈ 4,2 * 10^14`, vừa trong số nguyên 64 bit (Python không tràn dù sao).

In [15]:
POW_PRODUCTS = {prime_product(1 << i) for i in range(34)}

def reordered_prime_product(n: int) -> bool:
    return prime_product(n) in POW_PRODUCTS

for n, expected in examples + extra:
    assert reordered_prime_product(n) == expected
print("dấu vân tay tích nguyên tố OK")
print("tích lớn nhất có thể (10 chữ số 9):", 29 ** 10)

dấu vân tay tích nguyên tố OK
tích lớn nhất có thể (10 chữ số 9): 420707233300201


**Code nộp LeetCode (Cách 5):**

In [16]:
class Solution:
    def reorderedPowerOf2(self, n: int) -> bool:
        primes = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29)

        def sig(x: int) -> int:
            prod = 1
            while x:
                x, d = divmod(x, 10)
                prod *= primes[d]
            return prod

        return sig(n) in {sig(1 << i) for i in range(34)}

sol_product = Solution().reorderedPowerOf2
print([sol_product(n) for n, _ in examples + extra])

[True, False, True, True, False]


**Độ phức tạp (Cách 5):** tiền xử lý `O(34 * 10)`; mỗi truy vấn `O(L)`; bộ nhớ `O(34)`.

## 8. TRACE từng bước

### 8.1. Cách 3 cho `n = 4201`

`target = f(4201) = [1, 1, 1, 0, 1, 0, 0, 0, 0, 0]` (một chữ số 0, một 1, một 2, một 4). Duyệt `i = 1, 2, 4, 8, ...`; lũy thừa `1024 = 2^10` có bảng đếm `[1, 1, 1, 0, 1, 0, ...]` ⇒ khớp ⇒ `true`. Hàm in các lũy thừa cùng độ dài đã so sánh.

In [17]:
def trace_count(n):
    target = digit_count(n)
    print(f"n = {n}: bảng đếm {target}")
    i, k = 1, 0
    while i <= 10 ** 9:
        c = digit_count(i)
        if len(str(i)) == len(str(n)):
            print(f"  2^{k:2} = {i:>10}: {c} {'<-- KHỚP' if c == target else ''}")
        if c == target:
            return True
        i <<= 1
        k += 1
    return False

assert trace_count(4201) is True
print()
assert trace_count(4202) is False

n = 4201: bảng đếm [1, 1, 1, 0, 1, 0, 0, 0, 0, 0]
  2^10 =       1024: [1, 1, 1, 0, 1, 0, 0, 0, 0, 0] <-- KHỚP

n = 4202: bảng đếm [1, 0, 2, 0, 1, 0, 0, 0, 0, 0]
  2^10 =       1024: [1, 1, 1, 0, 1, 0, 0, 0, 0, 0] 
  2^11 =       2048: [1, 0, 1, 0, 1, 0, 0, 0, 1, 0] 
  2^12 =       4096: [1, 0, 0, 0, 1, 0, 1, 0, 0, 1] 
  2^13 =       8192: [0, 1, 1, 0, 0, 0, 0, 0, 1, 1] 


### 8.2. Cách 2 (quay lui theo bảng đếm) cho `n = 10`

Bảng đếm `{0: 1, 1: 1}`. Chữ số đầu khác 0: `d = 1` -> còn `{0: 1}` -> chữ số kế `0` -> `value = 10`: không là lũy thừa của 2. Không còn lựa chọn khác (`d = 0` bị cấm ở vị trí đầu). Kết quả `false`.

In [18]:
def trace_dfs(n):
    cnt = digit_count(n)
    L = len(str(n))
    leaves = []

    def dfs(pos, value):
        if pos == L:
            leaves.append((value, is_pow2(value)))
            return is_pow2(value)
        for d in range(10):
            if cnt[d] == 0:
                continue
            if pos == 0 and d == 0:
                print("  " * pos + f"bỏ d=0 ở vị trí đầu")
                continue
            print("  " * pos + f"chọn d={d} -> value={value * 10 + d}")
            cnt[d] -= 1
            if dfs(pos + 1, value * 10 + d):
                return True
            cnt[d] += 1
        return False

    res = dfs(0, 0)
    print("lá đã xét:", leaves, "->", res)
    return res

assert trace_dfs(10) is False
print()
assert trace_dfs(46) is True

bỏ d=0 ở vị trí đầu
chọn d=1 -> value=1
  chọn d=0 -> value=10
lá đã xét: [(10, False)] -> False

chọn d=4 -> value=4
  chọn d=6 -> value=46
chọn d=6 -> value=6
  chọn d=4 -> value=64
lá đã xét: [(46, False), (64, True)] -> True


### 8.3. Cách 5: tích số nguyên tố của vài số

`prime_product(1024) = 2 * 3 * 5 * 11` (chữ số 1 -> 3, 0 -> 2, 2 -> 5, 4 -> 11) = `330`; `prime_product(4201)` cũng bằng `330`; `prime_product(4202)` = `11 * 5 * 2 * 5 = 550`.

In [19]:
for x in (1024, 4201, 4202, 46, 64):
    prod = prime_product(x)
    digits = [int(c) for c in str(x)]
    print(f"{x:5}: các số nguyên tố {[PRIMES[d] for d in digits]} -> tích {prod:>5} | ở bảng lũy thừa? {prod in POW_PRODUCTS}")
assert prime_product(1024) == prime_product(4201) == 330

 1024: các số nguyên tố [3, 2, 5, 11] -> tích   330 | ở bảng lũy thừa? True
 4201: các số nguyên tố [11, 5, 2, 3] -> tích   330 | ở bảng lũy thừa? True
 4202: các số nguyên tố [11, 5, 2, 5] -> tích   550 | ở bảng lũy thừa? False
   46: các số nguyên tố [11, 17] -> tích   187 | ở bảng lũy thừa? True
   64: các số nguyên tố [17, 11] -> tích   187 | ở bảng lũy thừa? True


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (so bảng đếm với từng lũy thừa của 2): ngắn, `O(log^2 M)`, không phụ thuộc số chữ số; Cách 4 và 5 nhanh hơn một chút khi có nhiều truy vấn.

```python
class Solution:
    def reorderedPowerOf2(self, n: int) -> bool:
        def f(x: int) -> list:
            cnt = [0] * 10
            while x:
                x, v = divmod(x, 10)
                cnt[v] += 1
            return cnt

        target = f(n)
        i = 1
        while i <= 10**9:
            if f(i) == target:
                return True
            i <<= 1
        return False
```

In [20]:
class Solution:
    def reorderedPowerOf2(self, n: int) -> bool:
        def f(x: int) -> list:
            cnt = [0] * 10
            while x:
                x, v = divmod(x, 10)
                cnt[v] += 1
            return cnt

        target = f(n)
        i = 1
        while i <= 10**9:
            if f(i) == target:
                return True
            i <<= 1
        return False

sol = Solution()
for n, expected in examples + extra:
    got = sol.reorderedPowerOf2(n)
    print(n, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

1 -> True OK
10 -> False OK
46 -> True OK
4201 -> True OK
3 -> False OK


## 10. Vì sao đúng

**(a) Hoán vị ⇔ cùng đa tập chữ số.** Nếu `m` là hoán vị chữ số của `n` thì mỗi chữ số xuất hiện cùng số lần trong cả hai. Ngược lại, nếu hai số có cùng số lần xuất hiện của mỗi chữ số (thì cùng độ dài) ta xếp các chữ số của `n` thành đúng các chữ số của `m` theo thứ tự (chọn hoán vị biến `n` thành `m`). Khi `m` là lũy thừa của 2 (biểu diễn thập phân chuẩn, chữ số đầu khác 0), hoán vị đó **hợp lệ** (chữ số đầu của nó là chữ số đầu của `m`, khác 0).

**(b) Chỉ cần xét lũy thừa của 2 trong khoảng.** `n <= 10^9 < 2^30 = 1073741824` có tối đa 10 chữ số. Hoán vị của `n` có cùng độ dài (với chữ số đầu khác 0) nên nằm trong `[10^(L-1), 10^L)` với `L <= 10`, tức `< 10^10 < 2^34`. Do đó chỉ cần duyệt `2^0 .. 2^33`; cận `i <= 10^9` ở Cách 3 đủ vì hoán vị cùng độ dài 10 chữ số chỉ xảy ra khi `n = 10^9` (chữ số `1` và chín số `0`); hoán vị hợp lệ duy nhất là chính `10^9`, không là lũy thừa của 2; lũy thừa 10 chữ số đầu tiên là `2^30 = 1073741824` có chữ số khác. (Cách 4 và 5 dùng `range(34)` nên bao hết.)

**(c) Cách 2** liệt kê mọi số khác nhau có đa tập chữ số của `n` và chữ số đầu khác 0: đúng định nghĩa.

**(d) Cách 4.** Chuỗi chữ số đã sắp xếp xác định duy nhất đa tập chữ số (và ngược lại).

**(e) Cách 5.** Nếu hai đa tập chữ số khác nhau thì tích số nguyên tố khác nhau: phân tích ra thừa số nguyên tố của tích cho lại số lần mỗi số nguyên tố, tức số lần của mỗi chữ số; hai tích bằng nhau suy ra cùng đa tập.

**Kiểm chứng bằng máy:** dựng oracle **độc lập** bằng cách **sinh mọi hoán vị hợp lệ của từng lũy thừa của 2 có tối đa 8 chữ số** (tập các số "đáp án có"), rồi so mọi cách với oracle đó trên **mọi** `n` từ 1 tới `10^6` (và trên nhiều `n` ngẫu nhiên tới `10^8` với Cách 1 vét cạn khi số chữ số khác nhau không quá nhiều).

In [21]:
YES = set()
for i in range(34):
    p = str(1 << i)
    if len(p) > 8:
        continue
    for perm in set(permutations(p)):
        if perm[0] != "0":
            YES.add(int("".join(perm)))
print("số n (<= 8 chữ số) có đáp án true:", len(YES), "| lớn nhất:", max(YES))

def oracle(n: int) -> bool:
    return n in YES                              # chỉ hợp lệ khi n < 10^8

solutions = {
    "dfs": reordered_dfs,
    "count": reordered_count,
    "sorted": reordered_sorted_set,
    "product": reordered_prime_product,
    "sol_dfs": sol_dfs,
    "sol_count": sol_count,
    "sol_sorted": sol_sorted,
    "sol_product": sol_product,
    "final": Solution().reorderedPowerOf2,
}
fast = ["count", "sorted", "product", "sol_count", "sol_sorted", "sol_product", "final"]

for n in range(1, 10 ** 6 + 1):
    ref = n in YES
    assert reordered_sorted_set(n) == ref and reordered_prime_product(n) == ref, n
    if n % 7 == 0 or n < 3000:
        for name in fast:
            assert solutions[name](n) == ref, (name, n)
print("n = 1..10^6: các cách khớp oracle (mọi n cho hai cách nhanh nhất, mẫu cho các cách còn lại)")

số n (<= 8 chữ số) có đáp án true: 20596 | lớn nhất: 88766410


n = 1..10^6: các cách khớp oracle (mọi n cho hai cách nhanh nhất, mẫu cho các cách còn lại)


## 11. Time / Space Complexity

Gọi `L` là số chữ số của `n` (`<= 10`), `M = 10^9`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi hoán vị | `O(L! * L)` | `O(L!)` (set) |
| 2. Quay lui theo bảng đếm | `O(số số khác nhau * L)`, tệ nhất `~10!` | `O(L)` |
| 3. So bảng đếm với 30 lũy thừa | `O(log^2 M)` | `O(1)` |
| 4. Tập chuỗi sắp xếp | `O(L log L)` mỗi truy vấn (tiền xử lý một lần) | `O(34)` |
| 5. Tích số nguyên tố | `O(L)` mỗi truy vấn | `O(34)` |

Đo thử trên các `n` khác nhau (bỏ vét cạn cho 10 chữ số):

In [22]:
rng = random.Random(869)
tests = [rng.randint(1, 10 ** 9) for _ in range(20000)]
tests += [10 ** 9, 1234567890, 9876543210 % 10 ** 9, 536870912, 4201, 1024, 2048, 999999999]
ref = [reordered_sorted_set(n) for n in tests]
for name in ["count", "sorted", "product"]:
    fn = solutions[name]
    t0 = time.perf_counter()
    res = [fn(n) for n in tests]
    dt = (time.perf_counter() - t0) * 1000
    assert res == ref, name
    print(f"{name:8} {len(tests)} truy vấn: {dt:8.1f} ms")
print("số n có đáp án true trong mẫu:", sum(ref))
t0 = time.perf_counter()
d = reordered_dfs(234567891)
print(f"quay lui theo bảng đếm cho 234567891 ({d}): {1000 * (time.perf_counter() - t0):.0f} ms")

count    20008 truy vấn:    469.4 ms
sorted   20008 truy vấn:     20.9 ms
product  20008 truy vấn:     21.5 ms
số n có đáp án true trong mẫu: 11


quay lui theo bảng đếm cho 234567891 (False): 506 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với oracle, và test ngẫu nhiên có seed (đúng ràng buộc `1 <= n <= 10^9`).

In [23]:
# 12.1 ví dụ đề
for n, expected in examples + extra:
    for name, fn in solutions.items():
        assert fn(n) == expected, (name, n)
    assert reordered_brute(n) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    (1, True),                   # 2^0
    (2, True),
    (3, False),
    (4, True),
    (5, False),
    (8, True),
    (10, False),                 # 01 có chữ số 0 đứng đầu
    (16, True),
    (61, True),                  # hoán vị của 16
    (46, True),                  # 64
    (100, False),                # mọi hoán vị hợp lệ chỉ có 100
    (128, True),
    (182, True),                 # hoán vị của 128
    (1024, True),
    (4201, True),
    (2014, True),                # hoán vị của 1024
    (1000, False),               # chữ số 0 nhiều
    (10 ** 9, False),            # 1 và chín số 0
    (536870912, True),           # 2^29
    (123456789, False),          # 9 chữ số khác nhau không là hoán vị của lũy thừa nào
    (999999999, False),
]
for n, expected in edge:
    if n < 10 ** 8:
        assert (n in YES) == expected, ("oracle khác kỳ vọng", n, expected, n in YES)
    for name, fn in solutions.items():
        assert fn(n) == expected, (name, n, fn(n), expected)
    if len(str(n)) <= 8:
        assert reordered_brute(n) == expected, ("vét cạn khác kỳ vọng", n)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK


12.2 edge case: OK 21 ca


In [24]:
# 12.3 phản ví dụ: các cách sai (và tìm bằng chứng bằng tìm kiếm có seed)
def wrong_leading_zero(n):                 # cho phép chữ số 0 đứng đầu
    for perm in set(permutations(str(n))):
        if is_pow2(int("".join(perm))):
            return True
    return False

def wrong_only_smaller(n):                 # chỉ xét lũy thừa <= n
    target = digit_count(n)
    i = 1
    while i <= n:
        if digit_count(i) == target:
            return True
        i <<= 1
    return False

def wrong_set_not_multiset(n):             # so tập chữ số thay vì đa tập
    pow_sets = {frozenset(str(1 << i)) for i in range(34)}
    return frozenset(str(n)) in pow_sets

def wrong_digit_sum(n):                    # dấu vân tay quá yếu: (độ dài, tổng chữ số)
    pow_sigs = {(len(str(1 << i)), sum(map(int, str(1 << i)))) for i in range(34)}
    return (len(str(n)), sum(map(int, str(n)))) in pow_sigs

def wrong_few_powers(n):                   # chỉ xét lũy thừa <= 2^30 nhưng so sánh nhầm độ dài: dùng range(20)
    return digit_count(n) in [digit_count(1 << i) for i in range(20)]

def first_failure(wrong, limit=10 ** 5):
    for n in range(1, limit):
        if wrong(n) != (n in YES):
            return n
    return None

wrongs = [
    ("cho phép số 0 đứng đầu", wrong_leading_zero, 10),
    ("chỉ xét lũy thừa <= n", wrong_only_smaller, 46),
    ("so tập thay vì đa tập chữ số", wrong_set_not_multiset, None),
    ("dấu vân tay = tổng chữ số", wrong_digit_sum, None),
    ("thiếu các lũy thừa lớn", wrong_few_powers, 1048576),
]
for name, fn, n in wrongs:
    if n is None:
        n = first_failure(fn, 10 ** 6)
    assert n is not None, name
    print(f"{name:32} n={n}: sai = {fn(n)}, đúng = {n in YES}")
    assert fn(n) != (n in YES), name
print("12.3 phản ví dụ: OK (mỗi cách sai đều có n lộ lỗi)")

cho phép số 0 đứng đầu           n=10: sai = True, đúng = False
chỉ xét lũy thừa <= n            n=46: sai = False, đúng = True
so tập thay vì đa tập chữ số     n=11: sai = True, đúng = False
dấu vân tay = tổng chữ số        n=14: sai = True, đúng = False
thiếu các lũy thừa lớn           n=1048576: sai = False, đúng = True
12.3 phản ví dụ: OK (mỗi cách sai đều có n lộ lỗi)


In [25]:
# 12.4 ngẫu nhiên có seed, đúng ràng buộc 1 <= n <= 10^9
rng = random.Random(2869)
ns = [rng.randint(1, 10 ** 9) for _ in range(1500)]
ns += [rng.randint(1, 10 ** 4) for _ in range(500)]
# thêm nhiều n chắc chắn true: hoán vị ngẫu nhiên của một lũy thừa của 2
true_ns = []
for _ in range(500):
    p = str(1 << rng.randint(0, 29))
    digits = list(p)
    rng.shuffle(digits)
    if digits[0] == "0":
        continue
    true_ns.append(int("".join(digits)))
ns += true_ns
for n in ns:
    ref = reordered_sorted_set(n)
    for name in ["count", "product", "sol_count", "sol_sorted", "sol_product", "final"]:
        assert solutions[name](n) == ref, (name, n)
for n in true_ns:
    assert Solution().reorderedPowerOf2(n) is True, n                   # chắc chắn true theo cách xây
print("12.4 ngẫu nhiên và hoán vị của lũy thừa 2: OK (", len(ns), "giá trị )")

# 12.5 so với vét cạn (Cách 1) trên n ngẫu nhiên có ít chữ số khác nhau (<= 8 chữ số)
rng = random.Random(5)
for _ in range(150):
    L = rng.randint(1, 8)
    n = rng.randint(10 ** (L - 1), 10 ** L - 1)
    assert reordered_brute(n) == Solution().reorderedPowerOf2(n), n
print("12.5 so với vét cạn trên n <= 8 chữ số: OK")

# 12.6 tính chất: kết quả bất biến khi hoán vị chữ số của n (với hoán vị bắt đầu bằng chữ số khác 0)
rng = random.Random(6)
for _ in range(1000):
    n = rng.randint(1, 10 ** 9)
    digits = list(str(n))
    rng.shuffle(digits)
    if digits[0] == "0":
        continue
    assert Solution().reorderedPowerOf2(int("".join(digits))) == Solution().reorderedPowerOf2(n)
print("12.6 bất biến theo hoán vị chữ số: OK")

12.4 ngẫu nhiên và hoán vị của lũy thừa 2: OK ( 2458 giá trị )
12.5 so với vét cạn trên n <= 8 chữ số: OK
12.6 bất biến theo hoán vị chữ số: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cho phép chữ số 0 đứng đầu

`10` có hoán vị `01 = 1 = 2^0`. Nhưng đề cấm chữ số 0 đứng đầu.

In [26]:
print("sai (cho phép 0 đứng đầu):", wrong_leading_zero(10), "| đúng:", Solution().reorderedPowerOf2(10))
assert wrong_leading_zero(10) is True and Solution().reorderedPowerOf2(10) is False

sai (cho phép 0 đứng đầu): True | đúng: False


Vì sao sai: `int("01")` bằng `1`, một lũy thừa của 2; nhưng hoán vị `"01"` bắt đầu bằng 0. **Đúng:** bỏ các hoán vị có `perm[0] == "0"` (hoặc so với lũy thừa có chữ số đầu khác 0).

### Lỗi 2 — Chỉ xét lũy thừa của 2 nhỏ hơn hoặc bằng `n`

Hoán vị của `n` có thể **lớn hơn** `n`: `46 -> 64`.

In [27]:
print("sai (i <= n):", wrong_only_smaller(46), "| đúng:", Solution().reorderedPowerOf2(46))
assert wrong_only_smaller(46) is False and Solution().reorderedPowerOf2(46) is True

sai (i <= n): False | đúng: True


Vì sao sai: `64 > 46` nên vòng `while i <= n` dừng ở `32` trước khi tới `64`. **Đúng:** duyệt tới giới hạn của đề (`10^9`, hoặc `2^33`).

### Lỗi 3 — So **tập** chữ số thay vì **đa tập**

`set("122")` bằng `set("12")`: số lần xuất hiện bị bỏ qua.

In [28]:
n = first_failure(wrong_set_not_multiset, 10 ** 6)
print("n =", n, "| sai (so tập):", wrong_set_not_multiset(n), "| đúng:", Solution().reorderedPowerOf2(n))
assert wrong_set_not_multiset(n) != Solution().reorderedPowerOf2(n)
print("tập chữ số của", n, ":", sorted(set(str(n))), "| đa tập:", sorted(str(n)))

n = 11 | sai (so tập): True | đúng: False
tập chữ số của 11 : ['1'] | đa tập: ['1', '1']


Vì sao sai: các chữ số lặp lại làm hai đa tập khác nhau có cùng tập. **Đúng:** so bảng đếm `Counter` hoặc chuỗi đã sắp xếp.

### Lỗi 4 — Dấu vân tay quá yếu (tổng chữ số)

Hai số khác đa tập vẫn có thể có cùng tổng chữ số.

In [29]:
n = first_failure(wrong_digit_sum, 10 ** 6)
print("n =", n, "| sai (tổng chữ số):", wrong_digit_sum(n), "| đúng:", Solution().reorderedPowerOf2(n))
assert wrong_digit_sum(n) != Solution().reorderedPowerOf2(n)

n = 14 | sai (tổng chữ số): True | đúng: False


Vì sao sai: `n = 14` có độ dài 2 và tổng chữ số 5, giống `32`; nhưng `14` và `41` không là lũy thừa của 2. **Đúng:** dấu vân tay phải xác định duy nhất đa tập (đếm, sắp xếp, hoặc tích số nguyên tố).

### Lỗi 5 — Thiếu các lũy thừa lớn

`n` có thể có tới 10 chữ số; lũy thừa tương ứng có thể lớn (tới `2^33`). Chỉ duyệt vài lũy thừa đầu bỏ sót.

In [30]:
n = 1048576                                   # 2^20, 7 chữ số
print("n =", n, "(chính là lũy thừa nào?)", [1 << i for i in range(34) if digit_count(1 << i) == digit_count(n)])
print("sai (chỉ 2^0..2^19):", wrong_few_powers(n), "| đúng:", Solution().reorderedPowerOf2(n))
assert wrong_few_powers(n) is False and Solution().reorderedPowerOf2(n) is True

n = 1048576 (chính là lũy thừa nào?) [1048576]
sai (chỉ 2^0..2^19): False | đúng: True


Vì sao sai: lũy thừa đích nằm ngoài khoảng đã duyệt. **Đúng:** duyệt mọi lũy thừa tới `10^9` (hoặc `2^33` cho an toàn).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "hoán vị các chữ số" + điều kiện thuộc một **tập nhỏ** (lũy thừa của 2, số nguyên tố cố định...) -> **đảo ngược cách nhìn**: duyệt tập nhỏ và so "dấu vân tay".
- **Dấu vân tay của hoán vị:** bảng đếm, chuỗi đã sắp xếp, hoặc tích số nguyên tố (nếu cần một số duy nhất).
- Lũy thừa của 2: `x & (x - 1) == 0` hoặc `1 << i`; mỗi độ dài thập phân có 3-4 lũy thừa.
- **Đừng quên ràng buộc đầu số:** chữ số đầu khác 0 (khi sinh hoán vị).
- **Khử trùng hoán vị** bằng `set` hoặc quay lui theo bảng đếm.
- Cùng họ: Valid Anagram (242), Group Anagrams (49), Permutation in String (567), Power of Two (231), Largest Number (179).
- Test nên có: `1`, `10`, `46`, `4201`, `10^9`, chữ số lặp, số có nhiều chữ số 0.

## 15. Biến thể

1. **Đếm số cách hoán vị** (số khác nhau, chữ số đầu khác 0) cho ra lũy thừa của 2 (bài tập 5).
2. **Cơ số khác:** Reordered Power of `b` (ví dụ lũy thừa của 3): cùng ý tưởng với tập lũy thừa của `b`.
3. **Lũy thừa nhỏ nhất/lớn nhất** có cùng chữ số với `n`.
4. Bài liên quan: Next Permutation (31), Largest Time for Given Digits (949).

Dưới đây là code cho biến thể 2 (cơ số bất kỳ) và 3, kiểm bằng oracle vét cạn.

In [31]:
def reordered_power_of(n: int, base: int) -> bool:
    sig = "".join(sorted(str(n)))
    p = 1
    while len(str(p)) <= len(str(n)):
        if "".join(sorted(str(p))) == sig:
            return True
        p *= base
    return False

def reordered_power_of_brute(n: int, base: int) -> bool:
    powers = set()
    p = 1
    while p < 10 ** (len(str(n)) + 1):
        powers.add(p)
        p *= base
    return any(perm[0] != "0" and int("".join(perm)) in powers for perm in set(permutations(str(n))))

def matching_powers(n: int) -> List[int]:
    sig = "".join(sorted(str(n)))
    return [1 << i for i in range(34) if "".join(sorted(str(1 << i))) == sig]

for base in (2, 3, 5):
    for n in range(1, 2000):
        assert reordered_power_of(n, base) == reordered_power_of_brute(n, base), (n, base)
assert reordered_power_of(46, 2) and reordered_power_of(10, 3) is False
print("lũy thừa của 2 cùng chữ số với 4201:", matching_powers(4201), "| với 46:", matching_powers(46))
print("Biến thể: OK")

lũy thừa của 2 cùng chữ số với 4201: [1024] | với 46: [64]
Biến thể: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi hoán vị | Định nghĩa trực tiếp | `O(L! L)` | `O(L!)` | Oracle, `L <= 8` |
| 2. Quay lui theo bảng đếm | Không sinh trùng | tệ nhất `~10!` | `O(L)` | Khi không biết ý tưởng đảo ngược |
| 3. So bảng đếm | Duyệt 30 lũy thừa | `O(log^2 M)` | `O(1)` | **Cách nộp chính** |
| 4. Tập chuỗi sắp xếp | Tra bảng băm | `O(L log L)` | `O(34)` | Nhiều truy vấn |
| 5. Tích số nguyên tố | Dấu vân tay là một số | `O(L)` | `O(34)` | Nhanh nhất; thể hiện định lý số học |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print(digit_count(1024))
print("".join(sorted(str(4201))))
print(is_pow2(64), is_pow2(96), is_pow2(1))
```

In [32]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `n = 182`: ghi chuỗi chữ số đã sắp xếp của `n`, lũy thừa của 2 có cùng độ dài khớp với nó, và kết luận (`True`/`False`).

In [33]:
# Bài 2: điền kết quả tính tay
hand_signature = None        # chuỗi đã sắp xếp
hand_power = None            # lũy thừa của 2 khớp
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành cách so bảng đếm.

In [34]:
def exercise_3_count(n: int) -> bool:
    def f(x: int) -> List[int]:
        cnt = [0] * 10
        # TODO 1: lặp khi x > 0: lấy chữ số cuối bằng divmod(x, 10) và tăng cnt[chữ số]
        return cnt

    target = f(n)
    i = 1
    # TODO 2: lặp khi i <= 10**9: nếu f(i) == target thì trả True; sau đó i <<= 1
    return None


# assert exercise_3_count(1) is True
# assert exercise_3_count(10) is False
# assert exercise_3_count(46) is True
# assert exercise_3_count(4201) is True

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 5, hãy tự viết `exercise_4_product(n)` bằng dấu vân tay tích các số nguyên tố.

In [35]:
def exercise_4_product(n: int) -> bool:
    # TODO
    pass


# assert exercise_4_product(1) is True
# assert exercise_4_product(10) is False
# assert exercise_4_product(46) is True
# assert exercise_4_product(4201) is True

### Bài 5 — Mini challenge: đếm số cách hoán vị (khó)

Viết `exercise_5_count_ways(n)` trả về **số số khác nhau** (không có chữ số 0 đứng đầu) là hoán vị chữ số của `n` **và** là lũy thừa của 2.

```python
exercise_5_count_ways(46)    -> 1      (chỉ 64)
exercise_5_count_ways(4201)  -> 1      (chỉ 1024)
exercise_5_count_ways(1)     -> 1
exercise_5_count_ways(10)    -> 0
```
Gợi ý: không cần liệt kê hoán vị; đếm các lũy thừa của 2 có cùng bảng đếm.

In [36]:
def exercise_5_count_ways(n: int) -> int:
    # TODO
    pass


# assert exercise_5_count_ways(46) == 1
# assert exercise_5_count_ways(4201) == 1
# assert exercise_5_count_ways(1) == 1
# assert exercise_5_count_ways(10) == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `digit_count` trả về list 10 phần tử đếm từng chữ số `0..9`.
- Mức 2: `1024` có chữ số `1, 0, 2, 4` mỗi cái một lần; sắp xếp `4201` cho `0124`.
- Mức 3: `[1, 1, 1, 0, 1, 0, 0, 0, 0, 0]`, `0124`, `True False True`.

**Bài 2**
- Mức 1: các lũy thừa của 2 có 3 chữ số: `128, 256, 512`.
- Mức 2: chữ số của `182` sắp xếp: `128`.
- Mức 3: `"128"`, `128`, `True`.

**Bài 3**
- Mức 1: `while x: x, d = divmod(x, 10); cnt[d] += 1`.
- Mức 2: `while i <= 10**9: if f(i) == target: return True; i <<= 1`.
- Mức 3: cuối hàm `return False`.

**Bài 4**
- Mức 1: ánh xạ chữ số sang số nguyên tố `PRIMES[d]`.
- Mức 2: `prod = 1`; `while x: x, d = divmod(x, 10); prod *= PRIMES[d]`.
- Mức 3: `return prime_product(n) in {prime_product(1 << i) for i in range(34)}`.

**Bài 5**
- Mức 1: hai số cùng đa tập khi bảng đếm bằng nhau.
- Mức 2: đếm số `i` trong `1 << 0 .. 1 << 33` có `digit_count(i) == digit_count(n)`.
- Mức 3: mỗi lũy thừa chỉ tính một lần, và là một số **khác nhau** nên đếm bằng số lũy thừa khớp.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [37]:
print(digit_count(1024))
print("".join(sorted(str(4201))))
print(is_pow2(64), is_pow2(96), is_pow2(1))

assert digit_count(1024) == [1, 1, 1, 0, 1, 0, 0, 0, 0, 0]
assert "".join(sorted(str(4201))) == "0124"
assert (is_pow2(64), is_pow2(96), is_pow2(1)) == (True, False, True)

[1, 1, 1, 0, 1, 0, 0, 0, 0, 0]
0124
True False True


### Đáp án Bài 2

`"".join(sorted("182")) = "128"`; `128 = 2^7` có đúng chữ số đó; đáp án `True` (hoán vị `128`).

In [38]:
hand_signature_solution = "128"
hand_power_solution = 128
hand_answer_solution = True
assert "".join(sorted(str(182))) == hand_signature_solution
assert [p for p in powers_by_len[3] if "".join(sorted(str(p))) == hand_signature_solution] == [hand_power_solution]
assert Solution().reorderedPowerOf2(182) is hand_answer_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [39]:
def exercise_3_count_solution(n: int) -> bool:
    def f(x: int) -> List[int]:
        cnt = [0] * 10
        while x:
            x, d = divmod(x, 10)
            cnt[d] += 1
        return cnt

    target = f(n)
    i = 1
    while i <= 10 ** 9:
        if f(i) == target:
            return True
        i <<= 1
    return False

assert exercise_3_count_solution(1) is True
assert exercise_3_count_solution(10) is False
assert exercise_3_count_solution(46) is True
assert exercise_3_count_solution(4201) is True
rng = random.Random(3)
for _ in range(500):
    n = rng.randint(1, 10 ** 8)
    assert exercise_3_count_solution(n) == (n in YES), n
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [40]:
def exercise_4_product_solution(n: int) -> bool:
    primes = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29)

    def sig(x: int) -> int:
        prod = 1
        while x:
            x, d = divmod(x, 10)
            prod *= primes[d]
        return prod

    return sig(n) in {sig(1 << i) for i in range(34)}

assert exercise_4_product_solution(1) is True
assert exercise_4_product_solution(10) is False
assert exercise_4_product_solution(46) is True
assert exercise_4_product_solution(4201) is True
rng = random.Random(4)
for _ in range(500):
    n = rng.randint(1, 10 ** 8)
    assert exercise_4_product_solution(n) == (n in YES), n
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [41]:
def exercise_5_count_ways_solution(n: int) -> int:
    target = digit_count(n)
    return sum(1 for i in range(34) if digit_count(1 << i) == target)

def count_ways_oracle(n: int) -> int:
    return sum(1 for s in {"".join(p) for p in permutations(str(n))} if s[0] != "0" and is_pow2(int(s)))

assert exercise_5_count_ways_solution(46) == 1
assert exercise_5_count_ways_solution(4201) == 1
assert exercise_5_count_ways_solution(1) == 1
assert exercise_5_count_ways_solution(10) == 0
rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 10 ** 7)
    assert exercise_5_count_ways_solution(n) == count_ways_oracle(n), n
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (so bảng đếm với các lũy thừa của 2)

```python
class Solution:
    def reorderedPowerOf2(self, n: int) -> bool:
        def f(x: int) -> list:                 # dấu vân tay: số lần xuất hiện của mỗi chữ số
            cnt = [0] * 10
            while x:
                x, v = divmod(x, 10)
                cnt[v] += 1
            return cnt

        target = f(n)
        i = 1
        while i <= 10**9:                      # mọi lũy thừa của 2 trong phạm vi
            if f(i) == target:
                return True
            i <<= 1
        return False
```

Công thức nhớ nhanh: `n` xếp lại được thành lũy thừa của 2 ⇔ **tồn tại lũy thừa của 2 có cùng đa tập chữ số với `n`**.

### Checklist trước khi submit

- [ ] So **đa tập** chữ số (bảng đếm/chuỗi sắp xếp/tích số nguyên tố), không chỉ tập hay tổng?
- [ ] Duyệt đủ lũy thừa của 2 (cả những số **lớn hơn** `n`), tới `10^9` hoặc `2^33`?
- [ ] Không để chữ số 0 đứng đầu khi liệt kê hoán vị?
- [ ] Không dùng vét cạn hoán vị cho `n` 9-10 chữ số?
- [ ] Đã thử: `1`, `10`, `46`, `4201`, `10^9`, số có nhiều chữ số 0?
- [ ] Tên method đúng `reorderedPowerOf2(self, n)` và trả `bool`?